In [1]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Documents" / "BSE_Climate_Policy"
source = project / "raw_data" / "oecd_patents_raw.csv"

country_codes = {
    "AUS", "CAN", "DEU", "ESP", "FRA", "GBR",
    "IRL", "ITA", "MEX", "NZL", "USA"
}
years = {str(year) for year in range(2000, 2020)}

parts = []

for chunk in pd.read_csv(
    source,
    usecols=[
        "REF_AREA", "TIME_PERIOD", "TECH",
        "UNIT_MEASURE", "SIZE", "SEX", "OBS_VALUE"
    ],
    dtype=str,
    chunksize=50000
):
    selected = chunk[
        chunk["REF_AREA"].isin(country_codes)
        & chunk["TIME_PERIOD"].isin(years)
        & chunk["TECH"].eq("ENV_PAT")
        & chunk["UNIT_MEASURE"].eq("PATN")
        & chunk["SIZE"].eq("TWO")
        & chunk["SEX"].eq("_T")
    ]
    parts.append(selected)

patents = pd.concat(parts, ignore_index=True)
patents = patents.rename(columns={
    "REF_AREA": "country_code",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "green_patents"
})[["country_code", "year", "green_patents"]]

patents["year"] = patents["year"].astype(int)
patents["green_patents"] = pd.to_numeric(patents["green_patents"])
patents = patents.sort_values(["country_code", "year"]).reset_index(drop=True)

print("행·열:", patents.shape)
print("결측치:", patents["green_patents"].isna().sum())
print("국가·연도 중복:", patents.duplicated(["country_code", "year"]).sum())
display(patents.head())

행·열: (220, 3)
결측치: 0
국가·연도 중복: 0


,country_code,year,green_patents
0,AUS,2000,207.750000
1,AUS,2001,214.650000
2,AUS,2002,220.100000
3,AUS,2003,235.042857
4,AUS,2004,306.633333


In [2]:
patents_file = project / "processed_data" / "oecd_green_patents_2000_2019.csv"
patents.to_csv(patents_file, index=False)

panel_file = (
    project / "processed_data"
    / "cpu_carbon_renew_gdp_trade_industry_intensity_panel_draft_2000_2019.csv"
)
panel = pd.read_csv(panel_file)

full_panel = panel.merge(
    patents,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

assert len(full_panel) == 220
assert full_panel["green_patents"].notna().all()

output_file = project / "processed_data" / "research_panel_draft_2000_2019.csv"
full_panel.to_csv(output_file, index=False)

print("저장 완료:", output_file)
print("데이터 크기:", full_panel.shape)
display(full_panel.head())

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/research_panel_draft_2000_2019.csv
데이터 크기: (220, 10)


,country_code,year,CPU,CarbonPrice_draft,renewable_share,gdp_per_capita,trade_pct_gdp,industry_pct_gdp,carbon_intensity,green_patents
0,AUS,2000,37.986378,0.0,8.4,45979.076565,40.863126,24.532457,0.412379,207.750000
1,AUS,2001,32.302196,0.0,8.4,46314.623530,44.133731,23.586695,0.411628,214.650000
2,AUS,2002,23.147430,0.0,8.7,47599.407929,41.381313,23.555171,0.405011,220.100000
3,AUS,2003,32.739329,0.0,7.2,48509.663218,40.134801,23.991726,0.392333,235.042857
4,AUS,2004,32.399758,0.0,6.7,50018.590448,36.950516,23.881909,0.390602,306.633333


In [3]:
policy_check = full_panel.groupby("country_code").apply(
    lambda group: pd.Series({
        "정책_1_연도": ", ".join(
            group.loc[group["CarbonPrice_draft"].eq(1), "year"].astype(str)
        ),
        "정책값_빈칸_연도": ", ".join(
            group.loc[group["CarbonPrice_draft"].isna(), "year"].astype(str)
        ),
    }),
    include_groups=False
)

display(policy_check)

,정책_1_연도,정책값_빈칸_연도
country_code,,
AUS,"2012, 2013",2014
CAN,2019,
DEU,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",
ESP,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",
FRA,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",
GBR,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",
IRL,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",
ITA,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",
MEX,"2014, 2015, 2016, 2017, 2018, 2019",


In [4]:
climate_parts = []

for chunk in pd.read_csv(
    source,
    usecols=[
        "REF_AREA", "TIME_PERIOD", "TECH",
        "UNIT_MEASURE", "SIZE", "SEX", "OBS_VALUE"
    ],
    dtype=str,
    chunksize=50000
):
    selected = chunk[
        chunk["REF_AREA"].isin(country_codes)
        & chunk["TIME_PERIOD"].isin(years)
        & chunk["TECH"].eq("CCM")
        & chunk["UNIT_MEASURE"].eq("PATN")
        & chunk["SIZE"].eq("TWO")
        & chunk["SEX"].eq("_T")
    ]
    climate_parts.append(selected)

climate_patents = pd.concat(climate_parts, ignore_index=True)
climate_patents = climate_patents.rename(columns={
    "REF_AREA": "country_code",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "climate_patents"
})[["country_code", "year", "climate_patents"]]

climate_patents["year"] = climate_patents["year"].astype(int)
climate_patents["climate_patents"] = pd.to_numeric(
    climate_patents["climate_patents"]
)
climate_patents = climate_patents.sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

print("행·열:", climate_patents.shape)
print("결측치:", climate_patents["climate_patents"].isna().sum())
display(climate_patents.head())

행·열: (220, 3)
결측치: 0


,country_code,year,climate_patents
0,AUS,2000,138.750000
1,AUS,2001,167.900000
2,AUS,2002,160.933333
3,AUS,2003,179.233333
4,AUS,2004,213.800000


In [5]:
climate_file = project / "processed_data" / "oecd_climate_patents_2000_2019.csv"
climate_patents.to_csv(climate_file, index=False)

panel_with_climate = full_panel.merge(
    climate_patents,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

assert len(panel_with_climate) == 220
assert panel_with_climate["climate_patents"].notna().all()

output_file = project / "processed_data" / "research_panel_draft_with_climate_patents_2000_2019.csv"
panel_with_climate.to_csv(output_file, index=False)

print("저장 완료:", output_file)
print("데이터 크기:", panel_with_climate.shape)
display(panel_with_climate[
    ["country_code", "year", "green_patents", "climate_patents"]
].head())

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/research_panel_draft_with_climate_patents_2000_2019.csv
데이터 크기: (220, 11)


,country_code,year,green_patents,climate_patents
0,AUS,2000,207.750000,138.750000
1,AUS,2001,214.650000,167.900000
2,AUS,2002,220.100000,160.933333
3,AUS,2003,235.042857,179.233333
4,AUS,2004,306.633333,213.800000


In [6]:
analysis_columns = [
    "CPU",
    "CarbonPrice_draft",
    "renewable_share",
    "carbon_intensity",
    "climate_patents",
    "gdp_per_capita",
    "trade_pct_gdp",
    "industry_pct_gdp",
]

print("전체 행:", len(panel_with_climate))
print(
    "분석 변수에 빈칸이 없는 행:",
    panel_with_climate[analysis_columns].notna().all(axis=1).sum()
)

print("\n국가별 분석 가능 연도 수:")
display(
    panel_with_climate.assign(
        usable=panel_with_climate[analysis_columns].notna().all(axis=1)
    ).groupby("country_code")["usable"].sum()
)

전체 행: 220
분석 변수에 빈칸이 없는 행: 206

국가별 분석 가능 연도 수:


country_code
AUS    19
CAN    16
DEU    19
ESP    20
FRA    19
GBR    20
IRL    17
ITA    20
MEX    16
NZL    20
USA    20
Name: usable, dtype: int64